# Memory Garbage Collection
> Two-phase cleanup: prune expired entries, then prune decayed entries.

`MemoryGarbageCollector` removes stale memories in two phases:
1. **Expired** -- entries past their TTL
2. **Decayed** -- entries whose retention score has fallen below a threshold

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.memory.gc import MemoryGarbageCollector, GCStats
from anchor.memory.decay import LinearDecay
from anchor.storage import InMemoryEntryStore
from anchor.models import MemoryEntry
from datetime import datetime, timedelta, timezone

## Populate a Store with Entries
We create entries with varying ages to observe which survive collection.

In [ ]:
store = InMemoryEntryStore()
now = datetime.now(timezone.utc)

entries = [
    MemoryEntry(
        entry_id="mem-fresh",
        content="I just learned Python basics.",
        created_at=now - timedelta(hours=1),
    ),
    MemoryEntry(
        entry_id="mem-day-old",
        content="Yesterday I set up my dev environment.",
        created_at=now - timedelta(hours=24),
    ),
    MemoryEntry(
        entry_id="mem-week-old",
        content="Last week I started the ML course.",
        created_at=now - timedelta(hours=168),
    ),
    MemoryEntry(
        entry_id="mem-month-old",
        content="A month ago I explored Anchor.",
        created_at=now - timedelta(hours=720),
    ),
    MemoryEntry(
        entry_id="mem-ancient",
        content="Long ago I tried assembly programming.",
        created_at=now - timedelta(hours=2160),
    ),
]

for entry in entries:
    store.put(entry)

print(f"Store contains {len(entries)} entries:")
for e in entries:
    age_hours = (now - e.created_at).total_seconds() / 3600
    print(f"  {e.entry_id:<16} age={age_hours:>7.0f}h  '{e.content[:40]}...'")

## Create the Garbage Collector
We use `LinearDecay` with a 24-hour half-life for aggressive cleanup.

In [ ]:
decay = LinearDecay(half_life_hours=24.0)
gc = MemoryGarbageCollector(store=store, decay=decay)

print(f"Decay model: LinearDecay (half_life={decay.half_life_hours}h)")

## Dry Run
Preview what would be collected without actually deleting anything.

In [ ]:
stats = gc.collect(retention_threshold=0.1, dry_run=True)

print("=== Dry Run Results ===")
print(f"  Expired pruned:   {stats.expired_pruned}")
print(f"  Decayed pruned:   {stats.decayed_pruned}")
print(f"  Total remaining:  {stats.total_remaining}")

## Actual Collection
Run GC for real and confirm the store size shrank.

In [ ]:
stats = gc.collect(retention_threshold=0.1, dry_run=False)

print("=== GC Results ===")
print(f"  Expired pruned:   {stats.expired_pruned}")
print(f"  Decayed pruned:   {stats.decayed_pruned}")
print(f"  Total remaining:  {stats.total_remaining}")

## Inspect Surviving Entries

In [ ]:
print("Entries that survived GC:")
print(f"  (stats.total_remaining = {stats.total_remaining})")
print()
print("Fresh entries with high retention scores survive.")
print("Old entries with low retention scores are pruned.")

## Key Takeaways
- `MemoryGarbageCollector` runs in two phases: expired then decayed
- Always run with `dry_run=True` first to preview impact
- `retention_threshold` controls how aggressively decayed entries are pruned
- Pair with `LinearDecay` or `EbbinghausDecay` to tune the forgetting curve
- Schedule GC periodically to keep your memory store lean

**Back to:** [Memory README](README.md)